# Feature Families + Binary Target Engineering Matrix

This notebook checks whether Quant Warehouse can combine reusable feature families with binary target-engineering labels. It focuses on joinability, target sparsity, actual-event positive rates, and family-level signal separation rather than Rank IC.

Targets covered here:

- FMP event pairs: congress buy/sell, insider buy/sell, analyst upgrades/downgrades, price target raises/cuts, earnings beats/misses.
- Oracle trade entry labels from stored prices: yearly top-1 long/short entries using the batched optimal-trade solver with `YE: [1]`. Execution prices match `optimal_trader`: buy at high, sell at low, short at low, cover at high.

Important rate convention:

- `rows` is full symbol/date coverage for the target column.
- `rate_rows` is the actual paired-event denominator.
- `positive_rate` for paired labels is computed on actual paired rows, e.g. congress buy / (congress buy + congress sell), not buy / all symbol-date rows.

The goal is to answer: for each feature family, which target families have enough actual joined rows to be worth deeper modeling?

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'quant_warehouse').exists():
    REPO_ROOT = next(parent for parent in Path.cwd().parents if (parent / 'quant_warehouse').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    evaluate_feature_target_matrix,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

In [2]:
RUN_STARTED = perf_counter()

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=100_000_000_000,
    start_date='2018-01-01',
    horizons=(20, 60),
    min_observations=120,
    max_features_per_family=50,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        'congress',
        'insider',
        'analyst_rating',
        'price_target',
        'earnings',
    ),
    oracle_trade_k_by_frequency={'YE': (1,)},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col='high',  # buy execution
    oracle_trade_long_exit_price_col='low',    # sell execution
    oracle_trade_short_entry_price_col='low',  # short execution
    oracle_trade_short_exit_price_col='high',  # cover execution
)

FEATURE_CONFIG, TARGET_CONFIG

WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(
    config=WAREHOUSE.config,
    backend=WAREHOUSE.backend,
    catalog=WAREHOUSE.catalog,
    fundamentals=WAREHOUSE.fundamentals,
    equity_calendar=WAREHOUSE.equity_calendar,
)

## Universe and Feature Families

The notebook starts with FMP equities screened through the OpenBB/FMP screener route, then keeps symbols with the locally stored sections required for the fundamental feature families.

In [3]:
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)

print(f'Universe source: {universe_source}')
print(f'Eligible symbols: {len(symbols)}')
print(symbols[:50])

eligibility.head(20)

Universe source: openbb:fmp
Eligible symbols: 117
('AAPL', 'ABBV', 'ABT', 'ADI', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'ANET', 'APH', 'APP', 'AVGO', 'AXP', 'BA', 'BAC', 'BKNG', 'BLK', 'BMY', 'BNY', 'BRK-A', 'BRK-B', 'BX', 'C', 'CAT', 'CDNS', 'COF', 'COP', 'COST', 'CRM', 'CRWD', 'CSCO', 'CVS', 'CVX', 'DE', 'DELL', 'DHR', 'DIS', 'FTNT', 'GE', 'GEV', 'GILD', 'GLW', 'GOOG', 'GOOGL', 'GS', 'HD', 'HONIV', 'HWM', 'IBKR', 'IBM')


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

print({
    'screened_symbols': len(symbols),
    'raw_feature_panel_rows': len(raw_feature_panel),
    'raw_features': raw_feature_metadata['feature'].nunique(),
    **feature_timings,
})

feature_diagnostics.sort_values(['status', 'rows'], ascending=[True, False]).head(20)

{'screened_symbols': 117, 'raw_feature_panel_rows': 236598, 'raw_features': 376, 'raw_panel_build_seconds': 10.09579457482323}


,symbol,status,rows,features,seconds
0,AAPL,ok,2130,376,0.268606
1,ABBV,ok,2130,376,0.083545
2,ABT,ok,2130,376,0.071608
3,ADI,ok,2130,376,0.074077
4,AMAT,ok,2130,376,0.072055
5,AMD,ok,2130,376,0.081202
6,AMGN,ok,2130,376,0.075372
7,AMZN,ok,2130,376,0.063296
8,ANET,ok,2130,376,0.079653
9,APH,ok,2130,376,0.080046


## Binary Event Targets

Event targets are loaded without remote refresh. Cached normalized event-pair data is used when present; missing families are built from already stored FMP historical sections.

In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics
    .loc[event_diagnostics['combined_rows'].gt(0), 'symbol']
    .sort_values()
    .tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel['symbol'].isin(event_symbols)].copy()
feature_metadata = raw_feature_metadata.copy()
selected_features, capped_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    feature_metadata,
    max_features=FEATURE_CONFIG.max_features_per_family,
)
feature_inventory = (
    capped_feature_metadata
    .groupby(['source', 'family'])
    .agg(feature_count=('feature', 'nunique'))
    .reset_index()
    .sort_values(['source', 'family'])
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[['symbol', 'date']],
    events,
    TARGET_CONFIG,
)

print({
    'screened_symbols': len(symbols),
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'feature_panel_rows_after_event_symbol_filter': len(feature_panel),
    'capped_features': len(selected_features),
    'event_target_rows': len(event_target_panel),
    'event_target_columns': len(event_target_metadata),
    'event_load_seconds': round(event_load_seconds, 3),
})

display(feature_inventory)
event_diagnostics.sort_values('combined_rows', ascending=False).head(20)

{'screened_symbols': 117, 'event_symbols': 116, 'event_rows': 8340, 'feature_panel_rows_after_event_symbol_filter': 236587, 'capped_features': 365, 'event_target_rows': 236587, 'event_target_columns': 10, 'event_load_seconds': 3.416}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,39


,symbol,cached_rows,historical_rows,combined_rows,event_families
68,MSFT,34,945,979,"(analyst_rating, congress, earnings, insider, ..."
0,AAPL,32,820,852,"(analyst_rating, congress, earnings, insider, ..."
73,NVDA,25,799,824,"(analyst_rating, congress, earnings, insider, ..."
7,AMZN,34,719,753,"(analyst_rating, congress, earnings, insider, ..."
43,GOOGL,34,581,615,"(analyst_rating, congress, earnings, insider, ..."
103,TSLA,32,456,488,"(analyst_rating, congress, earnings, insider, ..."
63,META,34,437,471,"(analyst_rating, congress, earnings, insider, ..."
53,JPM,0,34,34,"(earnings,)"
56,LLY,0,34,34,"(earnings,)"
54,KLAC,0,34,34,"(earnings,)"


In [6]:
oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)

target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
target_metadata = pd.concat([event_target_metadata, oracle_target_metadata], ignore_index=True)
target_summary = summarize_binary_targets(target_panel, target_metadata)

print({
    'oracle_target_rows': len(oracle_target_panel),
    'oracle_target_columns': len(oracle_target_metadata),
    'oracle_k': TARGET_CONFIG.oracle_trade_k_by_frequency,
    'oracle_seconds': round(oracle_seconds, 3),
    'combined_target_rows': len(target_panel),
    'combined_target_columns': target_metadata['target'].nunique(),
})

display(target_summary)

rate_audit = target_summary[[
    'target', 'target_family', 'rows', 'rate_rows', 'positive_rows', 'positive_rate', 'positive_symbols'
]].copy()
rate_audit['positive_rate_pct'] = (rate_audit['positive_rate'] * 100).round(2)
display(rate_audit.sort_values(['target_family', 'rate_rows', 'positive_rows'], ascending=[True, False, False]))


{'oracle_target_rows': 236587, 'oracle_target_columns': 3, 'oracle_k': {'YE': (1,)}, 'oracle_seconds': 1.714, 'combined_target_rows': 236587, 'combined_target_columns': 13}


,target,target_family,rows,rate_rows,positive_rows,positive_rate,positive_symbols,min_positive_date,max_positive_date
0,target_event_on__earnings_beat,event,236587,3583,3045,0.849846,116,2018-01-12,2026-06-03
1,target_event_on__congress_buy,event,236587,2496,1374,0.550481,7,2018-01-25,2026-06-05
2,target_event_on__congress_sell,event,236587,2496,1341,0.537260,7,2018-01-04,2026-06-16
3,target_event_on__earnings_miss,event,236587,3583,538,0.150154,100,2018-01-22,2026-05-28
4,target_event_on__insider_sell,event,236587,538,534,0.992565,7,2018-08-08,2026-06-18
5,target_event_on__analyst_upgrade,event,236587,268,214,0.798507,7,2024-04-04,2026-06-09
6,target_event_on__price_target_raise,event,236587,245,202,0.824490,7,2024-01-11,2026-06-22
7,target_event_on__analyst_downgrade,event,236587,268,68,0.253731,7,2024-04-03,2026-06-22
8,target_event_on__price_target_cut,event,236587,245,60,0.244898,7,2024-01-02,2026-06-09
9,target_event_on__insider_buy,event,236587,538,4,0.007435,2,2021-03-10,2026-02-18


,target,target_family,rows,rate_rows,positive_rows,positive_rate,positive_symbols,positive_rate_pct
0,target_event_on__earnings_beat,event,236587,3583,3045,0.849846,116,84.98
3,target_event_on__earnings_miss,event,236587,3583,538,0.150154,100,15.02
1,target_event_on__congress_buy,event,236587,2496,1374,0.550481,7,55.05
2,target_event_on__congress_sell,event,236587,2496,1341,0.537260,7,53.73
4,target_event_on__insider_sell,event,236587,538,534,0.992565,7,99.26
9,target_event_on__insider_buy,event,236587,538,4,0.007435,2,0.74
5,target_event_on__analyst_upgrade,event,236587,268,214,0.798507,7,79.85
7,target_event_on__analyst_downgrade,event,236587,268,68,0.253731,7,25.37
6,target_event_on__price_target_raise,event,236587,245,202,0.824490,7,82.45
8,target_event_on__price_target_cut,event,236587,245,60,0.244898,7,24.49


## Feature Family x Target Matrix

This matrix asks whether a model row can be formed for each pair. `mean_abs_smd` is a fast family-level separation check: larger values mean the positive and negative target rows look more different on average. It is not a full trading evaluation.

In [7]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    capped_feature_metadata,
    target_panel,
    target_metadata,
    min_rows=120,
    min_positive_rows=10,
    min_feature_coverage=0.5,
)

usable = matrix.query("status == 'usable'").copy()

print({
    'matrix_rows': len(matrix),
    'usable_pairs': len(usable),
    'training_panel_rows': len(training_panel),
    'elapsed_seconds': round(perf_counter() - RUN_STARTED, 3),
})

display(usable.head(50))

usable_rate_audit = usable[[
    'source', 'feature_family', 'target_family', 'target', 'rows', 'positive_rows', 'positive_rate', 'mean_abs_smd', 'status'
]].copy()
usable_rate_audit['positive_rate_pct'] = (usable_rate_audit['positive_rate'] * 100).round(2)
display(usable_rate_audit.sort_values(['target_family', 'target', 'mean_abs_smd'], ascending=[True, True, False]).head(80))


{'matrix_rows': 195, 'usable_pairs': 180, 'training_panel_rows': 236587, 'elapsed_seconds': 26.54}


,source,feature_family,target_family,target,feature_count,rows,positive_rows,positive_rate,mean_feature_coverage,mean_abs_smd,status
15,financetoolkit,ft_ratios_profitability,event,target_event_on__earnings_beat,14,3583,3045,0.849846,1.000000,0.199487,usable
16,fmp,fmp_daily_mcap_yield,event,target_event_on__earnings_beat,14,3583,3045,0.849846,0.999761,0.077658,usable
17,fmp,fmp_income_mcap,event,target_event_on__earnings_beat,31,3583,3045,0.849846,1.000000,0.076774,usable
18,financetoolkit,ft_ratios_valuation,event,target_event_on__earnings_beat,24,3583,3045,0.849846,0.993895,0.073549,usable
19,financetoolkit,ft_ratios_solvency,event,target_event_on__earnings_beat,15,3583,3045,0.849846,1.000000,0.072136,usable
20,financetoolkit,ft_growth_cash,event,target_event_on__earnings_beat,50,3583,3045,0.849846,0.740000,0.065668,usable
21,financetoolkit,ft_ratios_liquidity,event,target_event_on__earnings_beat,7,3583,3045,0.849846,1.000000,0.065090,usable
22,fmp,fmp_cash_mcap,event,target_event_on__earnings_beat,39,3583,3045,0.849846,1.000000,0.064693,usable
23,fmp,fmp_daily_ev_multiple,event,target_event_on__earnings_beat,7,3583,3045,0.849846,1.000000,0.061032,usable
24,fmp,fmp_daily_mcap_multiple,event,target_event_on__earnings_beat,14,3583,3045,0.849846,0.998365,0.059656,usable


,source,feature_family,target_family,target,rows,positive_rows,positive_rate,mean_abs_smd,status,positive_rate_pct
165,financetoolkit,ft_ratios_efficiency,event,target_event_on__analyst_downgrade,268,68,0.253731,0.484126,usable,25.37
166,fmp,fmp_balance_mcap,event,target_event_on__analyst_downgrade,268,68,0.253731,0.361037,usable,25.37
167,financetoolkit,ft_ratios_valuation,event,target_event_on__analyst_downgrade,268,68,0.253731,0.285837,usable,25.37
168,financetoolkit,ft_growth_income,event,target_event_on__analyst_downgrade,268,68,0.253731,0.285609,usable,25.37
169,financetoolkit,ft_ratios_solvency,event,target_event_on__analyst_downgrade,268,68,0.253731,0.277958,usable,25.37
...,...,...,...,...,...,...,...,...,...,...
105,financetoolkit,ft_ratios_profitability,event,target_event_on__earnings_miss,3583,538,0.150154,0.199487,usable,15.02
119,fmp,fmp_balance_mcap,event,target_event_on__earnings_miss,3581,537,0.149958,0.098486,usable,15.00
106,fmp,fmp_daily_mcap_yield,event,target_event_on__earnings_miss,3583,538,0.150154,0.077658,usable,15.02
107,fmp,fmp_income_mcap,event,target_event_on__earnings_miss,3583,538,0.150154,0.076774,usable,15.02


In [8]:
coverage_pivot = (
    matrix
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='target',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

positive_pivot = (
    usable
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='positive_rows',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

coverage_pivot

target_family,source,feature_family,event,oracle_trade
0,financetoolkit,ft_growth_balance,10,3
1,financetoolkit,ft_growth_cash,10,3
2,financetoolkit,ft_growth_income,10,3
3,financetoolkit,ft_ratios_efficiency,10,3
4,financetoolkit,ft_ratios_liquidity,10,3
5,financetoolkit,ft_ratios_profitability,10,3
6,financetoolkit,ft_ratios_solvency,10,3
7,financetoolkit,ft_ratios_valuation,10,3
8,fmp,fmp_balance_mcap,10,3
9,fmp,fmp_cash_mcap,10,3


In [9]:
best_by_target = (
    usable
    .sort_values(['target', 'mean_abs_smd', 'positive_rows'], ascending=[True, False, False])
    .groupby('target')
    .head(3)
    .reset_index(drop=True)
)

best_by_target

,source,feature_family,target_family,target,feature_count,rows,positive_rows,positive_rate,mean_feature_coverage,mean_abs_smd,status
0,financetoolkit,ft_ratios_efficiency,event,target_event_on__analyst_downgrade,5,268,68,0.253731,1.000000,0.484126,usable
1,fmp,fmp_balance_mcap,event,target_event_on__analyst_downgrade,50,268,68,0.253731,1.000000,0.361037,usable
2,financetoolkit,ft_ratios_valuation,event,target_event_on__analyst_downgrade,24,268,68,0.253731,1.000000,0.285837,usable
3,financetoolkit,ft_ratios_efficiency,event,target_event_on__analyst_upgrade,5,268,214,0.798507,1.000000,0.415605,usable
4,fmp,fmp_balance_mcap,event,target_event_on__analyst_upgrade,50,268,214,0.798507,1.000000,0.348729,usable
5,financetoolkit,ft_growth_income,event,target_event_on__analyst_upgrade,38,268,214,0.798507,0.763158,0.315559,usable
6,financetoolkit,ft_growth_income,event,target_event_on__congress_buy,38,2496,1374,0.550481,0.763158,0.064547,usable
7,fmp,fmp_daily_ev_yield,event,target_event_on__congress_buy,7,2496,1374,0.550481,1.000000,0.063387,usable
8,financetoolkit,ft_ratios_efficiency,event,target_event_on__congress_buy,5,2496,1374,0.550481,1.000000,0.062529,usable
9,fmp,fmp_daily_ev_yield,event,target_event_on__congress_sell,7,2496,1341,0.537260,1.000000,0.077379,usable


In [10]:
sparse_targets = target_summary.query('positive_rows < 10').copy()
usable_targets = usable['target'].nunique() if not usable.empty else 0
usable_feature_families = usable['feature_family'].nunique() if not usable.empty else 0
best_pairs = usable.sort_values('mean_abs_smd', ascending=False).head(10) if not usable.empty else usable

lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} screened FMP symbols with market cap >= ${FEATURE_CONFIG.market_cap_min:,.0f}; {len(event_symbols)} symbols had at least one loaded event pair and were used in the target matrix.',
    f'- Feature side: {capped_feature_metadata["family"].nunique()} capped feature families and {len(selected_features)} selected feature columns.',
    f'- Target side: {target_metadata["target"].nunique()} binary target columns across same-day event and oracle-trade families; oracle is limited to {TARGET_CONFIG.oracle_trade_k_by_frequency}.',
    f'- Joinability: {len(usable)} feature-family/target pairs are usable under the current thresholds, covering {usable_targets} target columns and {usable_feature_families} feature families. For paired event/oracle labels, matrix rows are actual paired rows, not all no-event dates.',
]

if not sparse_targets.empty:
    lines.append(f'- Sparse targets: {len(sparse_targets)} target columns have fewer than 10 positive rows and should not be trusted for modeling yet.')

if 'rate_rows' in target_summary.columns:
    event_rate_preview = target_summary.sort_values(['target_family', 'rate_rows'], ascending=[True, False]).head(8)
    lines.append('')
    lines.append('Largest actual-event denominators:')
    for row in event_rate_preview[['target', 'rate_rows', 'positive_rows', 'positive_rate']].to_dict('records'):
        rate_pct = row['positive_rate'] * 100 if pd.notna(row['positive_rate']) else float('nan')
        lines.append(f'- {row["target"]}: {int(row["positive_rows"])} positive / {int(row["rate_rows"])} paired rows ({rate_pct:.2f}%).')

if not best_pairs.empty:
    lines.append('')
    lines.append('Top family/target combinations by fast separation score:')
    for row in best_pairs[['source', 'feature_family', 'target', 'positive_rows', 'mean_abs_smd']].to_dict('records'):
        lines.append(
            f'- {row["source"]}.{row["feature_family"]} -> {row["target"]}: '
            f'{int(row["positive_rows"])} positives, mean_abs_smd={row["mean_abs_smd"]:.4f}'
        )

lines.extend([
    '',
    'Interpretation:',
    '',
    '- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.',
    '- Oracle-trade labels are sparse yearly top-1 entry labels from the batched optimal-trade solver; this keeps dataset size manageable while preserving the buy/sell oracle task shape.',
    '- Congress, insider, analyst, price-target, and earnings labels are same-day sparse events. Their positive rates now use only actual paired event rows, which is the right denominator for buy/sell or upgrade/downgrade questions.',
    '- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.',
])

display(Markdown('\n'.join(lines)))

## Written Analysis

- Universe: 117 screened FMP symbols with market cap >= $100,000,000,000; 116 symbols had at least one loaded event pair and were used in the target matrix.
- Feature side: 15 capped feature families and 365 selected feature columns.
- Target side: 13 binary target columns across same-day event and oracle-trade families; oracle is limited to {'YE': (1,)}.
- Joinability: 180 feature-family/target pairs are usable under the current thresholds, covering 12 target columns and 15 feature families. For paired event/oracle labels, matrix rows are actual paired rows, not all no-event dates.
- Sparse targets: 1 target columns have fewer than 10 positive rows and should not be trusted for modeling yet.

Largest actual-event denominators:
- target_event_on__earnings_beat: 3045 positive / 3583 paired rows (84.98%).
- target_event_on__earnings_miss: 538 positive / 3583 paired rows (15.02%).
- target_event_on__congress_buy: 1374 positive / 2496 paired rows (55.05%).
- target_event_on__congress_sell: 1341 positive / 2496 paired rows (53.73%).
- target_event_on__insider_sell: 534 positive / 538 paired rows (99.26%).
- target_event_on__insider_buy: 4 positive / 538 paired rows (0.74%).
- target_event_on__analyst_upgrade: 214 positive / 268 paired rows (79.85%).
- target_event_on__analyst_downgrade: 68 positive / 268 paired rows (25.37%).

Top family/target combinations by fast separation score:
- financetoolkit.ft_ratios_efficiency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.7885
- fmp.fmp_daily_mcap_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6343
- financetoolkit.ft_ratios_profitability -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6048
- fmp.fmp_daily_ev_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5730
- financetoolkit.ft_ratios_liquidity -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5198
- financetoolkit.ft_ratios_efficiency -> target_event_on__analyst_downgrade: 68 positives, mean_abs_smd=0.4841
- financetoolkit.ft_ratios_solvency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4755
- fmp.fmp_daily_ev_yield -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4711
- fmp.fmp_cash_mcap -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4547
- financetoolkit.ft_growth_balance -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4545

Interpretation:

- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.
- Oracle-trade labels are sparse yearly top-1 entry labels from the batched optimal-trade solver; this keeps dataset size manageable while preserving the buy/sell oracle task shape.
- Congress, insider, analyst, price-target, and earnings labels are same-day sparse events. Their positive rates now use only actual paired event rows, which is the right denominator for buy/sell or upgrade/downgrade questions.
- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.

## Written Analysis

- Universe: 117 screened FMP symbols with market cap >= $100,000,000,000; 116 symbols had at least one loaded event pair and were used in the target matrix.
- Feature side: 15 capped feature families and 365 selected feature columns.
- Target side: 13 binary target columns across same-day event and oracle-trade families; oracle is limited to {'YE': (1,)}.
- Joinability: 180 feature-family/target pairs are usable under the current thresholds, covering 12 target columns and 15 feature families. For paired event/oracle labels, matrix rows are actual paired rows, not all no-event dates.
- Sparse targets: 1 target columns have fewer than 10 positive rows and should not be trusted for modeling yet.

Largest actual-event denominators:
- target_event_on__earnings_beat: 3045 positive / 3583 paired rows (84.98%).
- target_event_on__earnings_miss: 538 positive / 3583 paired rows (15.02%).
- target_event_on__congress_buy: 1374 positive / 2496 paired rows (55.05%).
- target_event_on__congress_sell: 1341 positive / 2496 paired rows (53.73%).
- target_event_on__insider_sell: 534 positive / 538 paired rows (99.26%).
- target_event_on__insider_buy: 4 positive / 538 paired rows (0.74%).
- target_event_on__analyst_upgrade: 214 positive / 268 paired rows (79.85%).
- target_event_on__analyst_downgrade: 68 positive / 268 paired rows (25.37%).

Top family/target combinations by fast separation score:
- financetoolkit.ft_ratios_efficiency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.7885
- fmp.fmp_daily_mcap_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6343
- financetoolkit.ft_ratios_profitability -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6048
- fmp.fmp_daily_ev_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5730
- financetoolkit.ft_ratios_liquidity -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5198
- financetoolkit.ft_ratios_efficiency -> target_event_on__analyst_downgrade: 68 positives, mean_abs_smd=0.4841
- financetoolkit.ft_ratios_solvency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4755
- fmp.fmp_daily_ev_yield -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4711
- fmp.fmp_cash_mcap -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4547
- financetoolkit.ft_growth_balance -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4545

Interpretation:

- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.
- Oracle-trade labels are sparse yearly top-1 entry labels from the batched optimal-trade solver; this keeps dataset size manageable while preserving the buy/sell oracle task shape.
- Congress, insider, analyst, price-target, and earnings labels are same-day sparse events. Their positive rates now use only actual paired event rows, which is the right denominator for buy/sell or upgrade/downgrade questions.
- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.